# 30 — Why explainability: one score, many reasons

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/030-why-explainability.ipynb)

A model returns a single number, but a decision usually needs a reason: which input pushed the score up, which pulled it down, and by how much. In this notebook you open up a small logistic model, attribute its score to each feature, and see where that story stops being honest.

## 1. The numbers

A logistic model scores an applicant by multiplying each feature by its weight, adding a bias, then squashing the result into a probability. Two features, two weights, one number. Everything after this section is just bookkeeping on these same numbers.

In [ ]:
import numpy as np                                      # the array library, preinstalled on Colab

w = np.array([1.2, -0.8])                               # the two weights the model learned
b = -0.5                                                # the bias: a constant added to every score
x = np.array([2.0, 1.0])                                # one applicant: feature 1 = 2, feature 2 = 1

score = x @ w + b                                       # @ is the dot product; then add the bias
prob = 1 / (1 + np.exp(-score))                         # squash the score into a probability

print("features x =", x)                                # show the input
print("weights  w =", w)                                # show what the model believes
print("score =", round(score, 2), " probability =", round(prob, 3))   # 1.1 and 0.75

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads NumPy, the array library Colab already has.</li><li><code>w = ...</code> — one weight per feature; the sign says which way it pushes.</li><li><code>b = -0.5</code> — a constant shift applied after the dot product.</li><li><code>x = ...</code> — the applicant: two feature values.</li><li><code>score = x @ w + b</code> — the raw evidence before squashing.</li><li><code>prob = 1 / (1 + np.exp(-score))</code> — the logistic curve maps any score into 0–1.</li><li><code>print(...)</code> — three lines of output: inputs, weights, and the verdict.</li></ul>

</details>

## 2. Step by step

The `@` hides which feature did the work. Break the sum apart: each feature contributes its own weight times its own value, and the contributions plus the bias rebuild the score exactly.

In [ ]:
total = b                                                # start the sum at the bias, not at zero

for j in range(len(x)):                                  # visit feature 0, then feature 1
    piece = w[j] * x[j]                                  # this feature's push on the score
    total = total + piece                                # add it to the running total
    print(f"feature {j}: w={w[j]:+.2f} x x={x[j]:.2f} -> {piece:+.2f}")   # show the term

print("rebuilt score =", round(total, 2))                # 1.1, identical to section 1
print("matches section 1:", np.isclose(total, score))    # True: the split is exact

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>total = b</code> — the accumulator starts at the bias.</li><li><code>for j in range(len(x))</code> — one iteration per feature.</li><li><code>piece = w[j] * x[j]</code> — the local contribution of feature <code>j</code>.</li><li><code>total = total + piece</code> — add this feature's push to the score.</li><li><code>print(f"...")</code> — an f-string showing weight, value, and contribution.</li><li><code>np.isclose(...)</code> — checks the two routes agree up to rounding.</li></ul>

</details>

## 3. Plot: pushes and pulls

The same applicant gets a positive push from one feature and a negative push from the other. A bar chart of `w * x` is the local explanation: the score is the bias plus the height of the bars.

In [ ]:
import matplotlib.pyplot as plt                          # plotting library, preinstalled on Colab

pieces = w * x                                           # both local contributions at once
names = ["feature 0", "feature 1"]                       # one label per bar
colors = ["tab:green" if v > 0 else "tab:red" for v in pieces]   # green helps, red hurts

plt.figure(figsize=(5, 3))                               # a small figure
plt.bar(names, pieces, color=colors)                     # one bar per feature's contribution
plt.axhline(0, color="grey", linewidth=0.8)              # the neutral line
plt.ylabel("push on the score (w times x)")              # what the height means
plt.title("same model, opposite pushes for this applicant")   # the takeaway
plt.show()                                               # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — the plotting library.</li><li><code>pieces = w * x</code> — elementwise product: each feature's local contribution.</li><li><code>names = [...]</code> — labels for the two bars.</li><li><code>colors = [...]</code> — a comprehension: green for positive, red for negative.</li><li><code>plt.figure(...)</code> — a fresh figure.</li><li><code>plt.bar(...)</code> — draw the two contributions.</li><li><code>plt.axhline(0, ...)</code> — the line that separates helping from hurting.</li><li><code>plt.title(...)</code> — states what the reader should notice.</li></ul>

</details>

## 4. Interactive: move one feature

Drag the first feature and watch its bar grow or shrink. The bias is fixed, the second feature is fixed, so any change in the probability is entirely due to this one input.

In [ ]:
from ipywidgets import interact                          # sliders that rerun a function

def show(x0=2.0):                                        # x0 comes from the slider
    live = np.array([x0, 1.0])                           # feature 1 varies, feature 2 does not
    pieces = w * live                                    # local contributions for this input
    s = pieces.sum() + b                                 # rebuild the score
    p = 1 / (1 + np.exp(-s))                             # and the probability

    plt.figure(figsize=(5, 3))                           # fresh figure for each drag
    plt.bar(["feature 0", "feature 1"], pieces, color=["tab:green" if v > 0 else "tab:red" for v in pieces])   # signed bars
    plt.axhline(0, color="grey", linewidth=0.8)          # neutral line
    plt.ylim(-4, 4)                                      # fixed axes, so nothing jumps
    plt.ylabel("push on the score")                      # axis label
    plt.title(f"x0 = {x0:.1f} -> score {s:+.2f}, p = {p:.2f}")   # live readout
    plt.show()                                           # draw it

interact(show, x0=(-2.0, 3.0, 0.1))                      # attach the slider

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — Colab ships the widgets library.</li><li><code>def show(x0=2.0)</code> — the function that redraws one frame.</li><li><code>live = np.array([x0, 1.0])</code> — feature 1 follows the slider; feature 2 is pinned.</li><li><code>pieces = w * live</code> — recompute both contributions.</li><li><code>s = pieces.sum() + b</code> — the score is the sum of the bars plus the bias.</li><li><code>p = ...</code> — the logistic squash, for the readout.</li><li><code>plt.bar(...)</code> — redraw the two bars in the same style as section 3.</li><li><code>interact(show, x0=(...))</code> — the slider, its range and step.</li></ul>

</details>

## 5. When it breaks: global weights vs one applicant

Global feature importance ranks features by the size of their weights, and people read that ranking as a fact about every decision. It is not. A feature with a large weight can be irrelevant for one person, because what matters locally is weight times value.

In [ ]:
x_new = np.array([0.2, 2.0])                             # a second applicant: feature 1 is tiny
local = w * x_new                                        # their local contributions

order_global = np.argsort(-np.abs(w))                    # features sorted by |weight|, biggest first
order_local = np.argsort(-np.abs(local))                 # same sort, but for this applicant

print("global ranking (by |w|):        ", order_global)  # [0 1]: feature 0 looks most important
print("local ranking (by |w times x|): ", order_local)   # [1 0]: for this person, feature 1 is
print("global says feature", order_global[0], "matters most")   # the headline number
print("but for this applicant feature", order_local[0], "matters most")   # the contradiction

plt.figure(figsize=(6, 3))                               # room for four bars
plt.bar(["f0 global", "f1 global"], np.abs(w), color="tab:blue")        # model-level sizes
plt.bar(["f0 local", "f1 local"], np.abs(local), color="tab:orange")    # applicant-level sizes
plt.ylabel("size of the effect")                         # comparable units on purpose
plt.title("global weights hide what happens to one applicant")   # the takeaway
plt.show()                                               # render

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>x_new = ...</code> — a different applicant, with a small first feature.</li><li><code>local = w * x_new</code> — contributions for this applicant only.</li><li><code>order_global = np.argsort(-np.abs(w))</code> — ranks features by weight size, descending.</li><li><code>order_local = np.argsort(-np.abs(local))</code> — ranks the same features locally.</li><li><code>print(...)</code> — the two rankings disagree, and the labels say so in words.</li><li><code>plt.bar(...)</code> — the first call draws global sizes, the second local sizes.</li><li><code>plt.title(...)</code> — names the failure: a global story about one person.</li></ul>

</details>

## Your turn

Change `x_new` in section 5 to `[3.0, 0.1]` and predict which ranking flips before you rerun. Then drag the section 4 slider until the probability crosses 0.5 and note which bar changed sign.